# 🎚️ Orange Harmony — Separador de Stems

Este notebook separa **voz** e **instrumental** (Demucs) para o Orange Studio do app. O app envia o áudio pelo Firebase, este notebook separa e devolve — **você não precisa copiar nenhum link**.

## Como usar (leva 1 minuto)
1. *(Opcional, mas 5–10× mais rápido)* menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4**.
2. Menu **Ambiente de execução → Executar tudo**.
3. Na primeira vez ele pede o arquivo `firebase_service_account.json` (o mesmo de sempre) — escolha o arquivo.
4. Quando aparecer **🟢 Separador ligado**, volte ao app: o card do Estúdio fica verde sozinho.

> 💡 **Dica para nunca mais escolher o arquivo:** clique na 🔑 (Segredos) à esquerda → *Adicionar novo segredo* → nome `FIREBASE_JSON`, valor = **todo o conteúdo** do arquivo JSON → ative *Acesso ao notebook*.

Deixe a **última célula rodando** enquanto for usar o Estúdio. Para desligar, pare a célula.

## 1. Instalar (a 1ª vez leva 2–4 min)

In [1]:
!pip install -q firebase-admin lameenc soundfile pydub
!pip install -q demucs torch torchaudio torchcodec
print('Instalação concluída.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.6/247.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 60.5 MB/s eta 0:00:00
Instalação concluída.


## 2. Arquivos do separador

In [2]:
%%writefile /content/firebase_config.py
# ══════════════════════════════════════════════════════════════
# firebase_config.py — Inicialização do Firebase Admin (Firestore + Storage)
#
# Credenciais: mesma lógica do app.py original —
#   1) variável de ambiente GOOGLE_APPLICATION_CREDENTIALS_JSON (conteúdo JSON completo)
#   2) arquivo local firebase_service_account.json
#
# Bucket do Storage: variável de ambiente FIREBASE_STORAGE_BUCKET.
# Se não for definida, tenta adivinhar como "{project_id}.appspot.com"
# (formato clássico) — projetos criados a partir de meados de 2024 usam
# "{project_id}.firebasestorage.app" e PRECISAM da variável explícita.
# ══════════════════════════════════════════════════════════════
import os
import json

import firebase_admin
from firebase_admin import credentials, firestore, storage

_db = None
_bucket = None
_erro_inicializacao = None


def _carregar_credencial():
    """Devolve (cred, project_id) ou (None, None) se não achar nada configurado."""
    bruto = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS_JSON")
    if bruto:
        info = json.loads(bruto)
        return credentials.Certificate(info), info.get("project_id")

    caminho = "firebase_service_account.json"
    if os.path.exists(caminho):
        with open(caminho, "r", encoding="utf-8") as f:
            info = json.load(f)
        return credentials.Certificate(caminho), info.get("project_id")

    return None, None


def _inicializar():
    global _db, _bucket, _erro_inicializacao
    try:
        if not firebase_admin._apps:
            cred, project_id = _carregar_credencial()
            if cred is None:
                _erro_inicializacao = (
                    "Nenhuma credencial do Firebase encontrada. Defina a variável de "
                    "ambiente GOOGLE_APPLICATION_CREDENTIALS_JSON (conteúdo do JSON da "
                    "service account) ou coloque o arquivo firebase_service_account.json "
                    "na pasta do backend."
                )
                return

            bucket_nome = os.environ.get("FIREBASE_STORAGE_BUCKET") or (
                f"{project_id}.appspot.com" if project_id else None
            )
            firebase_admin.initialize_app(cred, {"storageBucket": bucket_nome} if bucket_nome else None)

        _db = firestore.client()
        try:
            _bucket = storage.bucket()
        except Exception as e:
            _erro_inicializacao = (
                f"Firestore conectou, mas o Storage não: {e}. Confira se você definiu "
                "FIREBASE_STORAGE_BUCKET com o nome exato do bucket (aparece no console "
                "do Firebase, em Storage)."
            )
    except Exception as e:
        _erro_inicializacao = f"Falha ao inicializar o Firebase: {e}"


_inicializar()


def get_db():
    return _db


def get_bucket():
    return _bucket


def get_erro():
    return _erro_inicializacao


Writing /content/firebase_config.py


In [3]:
%%writefile /content/audio_analysis.py
# ══════════════════════════════════════════════════════════════
# audio_analysis.py — Funções puras de análise vocal (Orange Harmony)
# Portado do app.py Streamlit: nenhuma dependência de UI aqui,
# só numpy/librosa. Pode ser testado e usado de qualquer front-end.
# ══════════════════════════════════════════════════════════════
import io
import gc
import shutil
import subprocess
import tempfile
import os
import threading
import numpy as np
import librosa

# ── Limites de recurso ──
# O plano grátis do Render tem só 512 MB de RAM e 0.1 CPU. Medido: sem esses
# limites, um áudio de 3 min pedia ~670 MB (o servidor era morto e reiniciava,
# derrubando até as mensagens de texto da Laranjinha).
LIMITE_ANALISE_S = 120      # analisa no máximo os primeiros 120 s do áudio
BLOCO_PITCH_S = 20          # o pyin roda em blocos de 20 s (memória constante)
JANELA_TOM_BPM_S = 60       # tom/BPM são estáveis: 60 s do meio bastam


def liberar_memoria():
    """Devolve memória ao sistema operacional depois de uma análise pesada.
    O Python solta os arrays, mas o alocador do Linux costuma ficar com a
    memória reservada — malloc_trim faz ela voltar (senão o pico vira permanente)."""
    gc.collect()
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def limitar_duracao(audio, sr, max_s=LIMITE_ANALISE_S):
    """Corta o áudio em `max_s` segundos. Devolve (audio, aviso) — aviso em ASCII
    (vai em header HTTP) ou None."""
    if audio is not None and len(audio) > int(max_s * sr):
        return audio[: int(max_s * sr)], f"cortado_{max_s}s"
    return audio, None


def _trecho_central(audio, sr, segundos):
    """Devolve até `segundos` do meio do áudio (evita intro/silêncio no começo)."""
    n = int(segundos * sr)
    if len(audio) <= n:
        return audio
    ini = (len(audio) - n) // 2
    return audio[ini:ini + n]


# ── Constantes musicais ──
NOMES_NOTAS = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
NOTAS_REFERENCIA = [f"{n}{o}" for o in range(2, 6) for n in NOMES_NOTAS]
ESCALA_MAIOR = [0, 2, 4, 5, 7, 9, 11, 12]


def f0_para_midi_calibrado(f0, calibracao=440.0):
    return 69 + 12 * np.log2(f0 / calibracao)


def f0_para_freq(nota, calibracao=440.0):
    """Converte um nome de nota (ex: 'C4') para frequência em Hz."""
    nome = nota[:-1]
    oitava = int(nota[-1])
    midi = 12 * (oitava + 1) + NOMES_NOTAS.index(nome)
    return calibracao * 2 ** ((midi - 69) / 12)


def _caminho_ffmpeg():
    """ffmpeg embutido via pip (imageio-ffmpeg) — não depende de o servidor ter
    ffmpeg instalado. Se não houver, tenta o ffmpeg do sistema."""
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return shutil.which("ffmpeg")


def _decodificar_com_ffmpeg(caminho, sr_alvo=22050):
    """Decodifica QUALQUER formato (WebM/Opus do Chrome, MP4/AAC do Safari, M4A...)
    direto pro ffmpeg, já em mono e 22050 Hz. Devolve (audio, sr) ou (None, None)."""
    exe = _caminho_ffmpeg()
    if not exe:
        return None, None
    try:
        proc = subprocess.run(
            [exe, "-v", "error", "-i", caminho, "-vn", "-ac", "1", "-ar", str(sr_alvo), "-f", "f32le", "-"],
            capture_output=True, timeout=90,
        )
        if proc.returncode != 0 or len(proc.stdout) < 8:
            return None, None
        return np.frombuffer(proc.stdout, dtype="<f4").copy(), sr_alvo
    except Exception:
        return None, None


def carregar_audio_bytes(dados: bytes, nome_arquivo: str = "audio.wav"):
    """Lê bytes de áudio (upload) e devolve (audio, sr) em 22050 Hz mono.
    Cadeia de tentativas, da mais rápida pra mais abrangente:
      1) soundfile — WAV, FLAC, OGG, MP3 (rápido, sem processo externo)
      2) ffmpeg embutido — WebM/Opus (gravador do Chrome), MP4/AAC (Safari/iPhone), M4A
      3) librosa.load  4) pydub
    O gravador do navegador já manda WAV (ver recorder.js), então na prática o
    passo 1 resolve; os outros cobrem arquivos enviados em outros formatos."""
    if not dados:
        return None, None
    ext = os.path.splitext(nome_arquivo)[1] or ".wav"
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        tmp.write(dados)
        tmp_path = tmp.name

    audio, sr = None, None
    try:
        try:
            import soundfile as sf
            audio, sr = sf.read(tmp_path, dtype="float32")
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            if sr != 22050:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=22050)
                sr = 22050
        except Exception:
            audio, sr = None, None

        if audio is None:
            audio, sr = _decodificar_com_ffmpeg(tmp_path)

        if audio is None:
            try:
                audio, sr = librosa.load(tmp_path, sr=22050, mono=True)
            except Exception:
                audio, sr = None, None

        if audio is None:
            try:
                from pydub import AudioSegment
                segmento = AudioSegment.from_file(tmp_path).set_frame_rate(22050).set_channels(1)
                amostras = np.array(segmento.get_array_of_samples(), dtype=np.float32)
                amostras /= float(1 << (8 * segmento.sample_width - 1))
                audio, sr = amostras, 22050
            except Exception:
                audio, sr = None, None
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass

    if audio is None or len(audio) == 0:
        return None, None
    return audio.astype(np.float32), int(sr)


# ══════════════════ PITCH ══════════════════
def extrair_pitch(audio, sr):
    """Pitch (F0) via pyin, processado em blocos de BLOCO_PITCH_S segundos.
    O pyin rodando no áudio inteiro gasta memória proporcional à duração
    (medido: 180 s -> 671 MB). Em blocos, o pico fica constante. Cada bloco tem
    um múltiplo exato do hop (512), então os tempos batem com a versão sem blocos;
    o último quadro de cada bloco (duplicado pelo `center=True`) é descartado."""
    hop = 512
    bloco = (int(BLOCO_PITCH_S * sr) // hop) * hop
    n = len(audio)

    limites = [(ini, min(ini + bloco, n)) for ini in range(0, n, bloco)]
    # um resto minúsculo no fim não dá pro pyin (precisa de > frame_length/2): funde no bloco anterior
    if len(limites) > 1 and (limites[-1][1] - limites[-1][0]) < 4096:
        ini_ult, fim_ult = limites.pop()
        ini_ant, _ = limites.pop()
        limites.append((ini_ant, fim_ult))

    f0_partes, voz_partes = [], []
    for i, (ini, fim) in enumerate(limites):
        f0_b, voiced_b, _ = librosa.pyin(audio[ini:fim], fmin=80, fmax=1000, sr=sr,
                                         frame_length=2048, hop_length=hop)
        if i < len(limites) - 1:          # quadro final = início do próximo bloco
            f0_b, voiced_b = f0_b[:-1], voiced_b[:-1]
        f0_partes.append(f0_b)
        voz_partes.append(voiced_b)

    f0 = np.concatenate(f0_partes)
    voiced = np.concatenate(voz_partes)
    tempos = librosa.times_like(f0, sr=sr, hop_length=hop)
    f0 = np.where(voiced & ~np.isnan(f0), f0, 0.0)
    return tempos, f0


def segmentar_notas(f0, tempos, duracao_min=0.4):
    mascara = f0 > 0
    if mascara.sum() == 0:
        return []
    mudancas = np.diff(mascara.astype(int))
    inicios = np.where(mudancas == 1)[0] + 1
    fins = np.where(mudancas == -1)[0] + 1
    if mascara[0]:
        inicios = np.concatenate(([0], inicios))
    if mascara[-1]:
        fins = np.concatenate((fins, [len(mascara)]))
    dt = tempos[1] - tempos[0] if len(tempos) > 1 else 0.01
    return [f0[i:f] for i, f in zip(inicios, fins) if (f - i) * dt >= duracao_min]


def analisar_afinacao(f0_limpo, tempos, calibracao=440.0, nota_ref=None):
    mascara = f0_limpo > 0
    vazio = {"nota_predominante": "—", "desvio_medio_cents": 0.0, "tendencia": "—",
             "desvio_sinal_cents": 0.0, "pct_afinado": 0.0, "num_frases": 0,
             "sustentacao_media": 0.0, "num_pausas": 0, "pausa_media": 0.0}
    if mascara.sum() == 0:
        return vazio
    f0_voz = f0_limpo[mascara]
    mediana_orig = float(np.median(f0_voz))
    f0_pred = mediana_orig
    for divisor in (2, 4):
        candidato = mediana_orig / divisor
        frac = float(np.mean(np.abs(f0_voz - candidato) / candidato < 0.08))
        if frac > 0.5:
            f0_pred = candidato
            break
    midi_pred = f0_para_midi_calibrado(f0_pred, calibracao)
    midi_arred_pred = int(round(midi_pred))
    nota_pred = f"{NOMES_NOTAS[midi_arred_pred % 12]}{midi_arred_pred // 12 - 1}"
    if nota_ref:
        f_ref = f0_para_freq(nota_ref, calibracao)
    else:
        f_ref = calibracao * 2 ** ((midi_arred_pred - 69) / 12)
    cents = 1200 * np.log2(f0_voz / f_ref)
    desvio_medio = float(np.mean(np.abs(cents)))
    desvio_sinal = float(np.mean(cents))
    pct_afinado = float(np.mean(np.abs(cents) <= 50) * 100)
    tendencia = ("neutra (bem centrada)" if abs(desvio_sinal) < 10
                 else ("aguda (sharp)" if desvio_sinal > 0 else "grave (flat)"))
    dt = tempos[1] - tempos[0] if len(tempos) > 1 else 0.01
    mudancas = np.diff(mascara.astype(int))
    inicios = np.where(mudancas == 1)[0] + 1
    fins = np.where(mudancas == -1)[0] + 1
    if mascara[0]:
        inicios = np.concatenate(([0], inicios))
    if mascara[-1]:
        fins = np.concatenate((fins, [len(mascara)]))
    frases = [(f - i) * dt for i, f in zip(inicios, fins) if (f - i) * dt >= 0.3]
    pausas = [(inicios[k + 1] - fins[k]) * dt for k in range(len(fins) - 1) if (inicios[k + 1] - fins[k]) * dt >= 0.3]
    return {"nota_predominante": nota_pred, "desvio_medio_cents": desvio_medio,
            "tendencia": tendencia, "desvio_sinal_cents": desvio_sinal,
            "pct_afinado": pct_afinado, "num_frases": len(frases),
            "sustentacao_media": float(np.mean(frases)) if frases else 0.0,
            "num_pausas": len(pausas), "pausa_media": float(np.mean(pausas)) if pausas else 0.0}


def extrair_sequencia_notas(f0, tempos, min_dur=0.15):
    """Detecta a sequência de notas sustentadas na performance (melodia cantada)."""
    notas = []
    atual, inicio = None, None
    for t, f in zip(tempos, f0):
        if f <= 0:
            if atual is not None and (t - inicio) >= min_dur:
                notas.append((atual, round(t - inicio, 2)))
            atual, inicio = None, None
            continue
        n = librosa.hz_to_note(f)
        if n != atual:
            if atual is not None and (t - inicio) >= min_dur:
                notas.append((atual, round(t - inicio, 2)))
            atual, inicio = n, t
    if atual is not None and (tempos[-1] - inicio) >= min_dur:
        notas.append((atual, round(tempos[-1] - inicio, 2)))
    return notas


# ══════════════════ VIBRATO ══════════════════
def detectar_vibrato_v4(f0, tempos, calibracao_a4=440.0, duracao_min=0.8):
    segmentos = segmentar_notas(f0, tempos, duracao_min=duracao_min)
    dt = tempos[1] - tempos[0] if len(tempos) > 1 else 0.01
    resultados = []
    for seg in segmentos:
        f0_nota = seg[seg > 0]
        if len(f0_nota) < 25:
            continue
        f0_mediana = float(np.median(f0_nota))
        cents_rel = 1200 * np.log2(f0_nota / f0_mediana)
        cents_rel = cents_rel - np.mean(cents_rel)
        n = len(cents_rel)
        t_axis = np.arange(n) * dt
        coef = np.polyfit(t_axis, cents_rel, 1)
        cents_detrended = cents_rel - np.polyval(coef, t_axis)
        deslize_cents = float(abs(coef[0]) * (n * dt))
        janela = np.hanning(n)
        espectro = np.abs(np.fft.rfft(cents_detrended * janela))
        freqs = np.fft.rfftfreq(n, d=dt)
        mascara = (freqs >= 3) & (freqs <= 8)
        if mascara.sum() == 0:
            continue
        idx_pico = np.argmax(espectro[mascara])
        taxa = float(freqs[mascara][idx_pico])
        energia_pico = espectro[mascara][idx_pico] ** 2
        energia_total = float(np.sum(espectro ** 2))
        periodicidade = energia_pico / energia_total if energia_total > 0 else 0.0
        fft_completo = np.fft.rfft(cents_detrended)
        faixa = (freqs >= taxa - 1.0) & (freqs <= taxa + 1.0)
        oscilacao = np.fft.irfft(fft_completo * faixa, n=n)
        extensao = float(2 * np.max(np.abs(oscilacao)))
        midi = f0_para_midi_calibrado(f0_mediana, calibracao_a4)
        midi_arredondado = int(np.round(midi))
        nome = NOMES_NOTAS[midi_arredondado % 12]
        oitava = midi_arredondado // 12 - 1
        resultados.append({"nota": f"{nome}{oitava}", "taxa_hz": taxa, "extensao_cents": extensao,
                            "deslize_cents": deslize_cents, "periodicidade": periodicidade,
                            "duracao_s": round(len(seg) * dt, 2)})
    return resultados


def classificar_vibrato_v4(taxa, extensao, deslize, periodicidade):
    if periodicidade < 0.15:
        return "deslize (pitch derrapou)" if deslize > 50 else "nota estável (sem vibrato)"
    if taxa < 3 or taxa > 8:
        return f"fora da faixa ({taxa:.1f} Hz)"
    if extensao < 15:
        return "fraco (pouca oscilação)"
    if extensao <= 60:
        return "saudável"
    if extensao <= 120:
        return "vibrato largo (expressivo)"
    return "vibrato muito largo"


# ══════════════════ BPM / TOM / COVER ══════════════════
def detectar_bpm_e_beats(audio, sr):
    try:
        # BPM é estável ao longo da música: analisa até JANELA_TOM_BPM_S*2 segundos
        # (as batidas depois disso são extrapoladas pela grade do BPM abaixo).
        trecho = audio[: int(JANELA_TOM_BPM_S * 2 * sr)]
        tempo, beat_frames = librosa.beat.beat_track(y=trecho, sr=sr)
        bpm = float(np.atleast_1d(tempo)[0])
        if bpm < 50 or bpm > 200 or np.isnan(bpm):
            bpm = 90.0
        beat_times = np.asarray(librosa.frames_to_time(beat_frames, sr=sr), dtype=np.float64)
        # Se o áudio é mais longo que a janela analisada, continua a grade de
        # batidas até o fim com o intervalo mediano (a Produção usa as batidas
        # pra posicionar baixo/bateria na música toda).
        duracao = len(audio) / sr
        if len(beat_times) >= 2 and duracao > len(trecho) / sr + 0.5:
            intervalo = float(np.median(np.diff(beat_times)))
            if intervalo > 0:
                extras = np.arange(beat_times[-1] + intervalo, duracao, intervalo)
                beat_times = np.concatenate([beat_times, extras])
        return round(bpm, 1), beat_times
    except Exception:
        return 90.0, np.array([])


def detectar_tom(audio, sr):
    try:
        chroma = librosa.feature.chroma_cqt(y=_trecho_central(audio, sr, JANELA_TOM_BPM_S),
                                            sr=sr, hop_length=1024)
        chroma_mean = chroma.mean(axis=1)
        idx = int(np.argmax(chroma_mean))
        return NOMES_NOTAS[idx]
    except Exception:
        return "C"


def analisar_cover(audio, sr, calibracao=440.0):
    resultado = {"tom": "—", "bpm": 0.0, "pct_na_escala": 0.0, "notas_fora": [],
                 "notas_principais": [], "veredito": "—", "duracao_s": 0.0}
    if audio is None or len(audio) < int(sr * 0.5):
        return resultado
    resultado["duracao_s"] = round(len(audio) / sr, 1)
    tom = detectar_tom(audio, sr)
    resultado["tom"] = tom
    bpm, _ = detectar_bpm_e_beats(audio, sr)
    resultado["bpm"] = bpm
    raiz = NOMES_NOTAS.index(tom)
    escala_pc = set((raiz + i) % 12 for i in ESCALA_MAIOR)
    tempos, f0 = extrair_pitch(audio, sr)
    f0_limpo = np.where((f0 >= 80) & (f0 <= 1000), f0, 0.0)
    mascara = f0_limpo > 0
    if mascara.sum() == 0:
        resultado["veredito"] = "Sem sinal de pitch detectado na gravação."
        return resultado
    f0_voz = f0_limpo[mascara]
    midi = f0_para_midi_calibrado(f0_voz, calibracao)
    pc = np.round(midi).astype(int) % 12
    dentro = np.isin(pc, list(escala_pc))
    pct = float(np.mean(dentro) * 100)
    resultado["pct_na_escala"] = round(pct, 1)
    contagem = {}
    for p in pc:
        contagem[p] = contagem.get(p, 0) + 1
    principais = sorted(contagem.items(), key=lambda x: -x[1])[:6]
    resultado["notas_principais"] = [NOMES_NOTAS[p] for p, _ in principais]
    fora = set(int(p) for p in pc[~dentro])
    resultado["notas_fora"] = [NOMES_NOTAS[p] for p in sorted(fora)]
    if pct >= 90:
        resultado["veredito"] = "Excelente — a gravação está casando com o tom"
    elif pct >= 75:
        resultado["veredito"] = "Bom — a maior parte está no tom, com alguns escapes"
    elif pct >= 60:
        resultado["veredito"] = "Regular — várias notas fora do tom detectado"
    else:
        resultado["veredito"] = "Fora do eixo — a gravação não está casando com o tom"
    return resultado


# ══════════════════ FUNÇÃO PRINCIPAL (usada pela rota /api/analyze) ══════════════════
def analisar_audio_completo(dados: bytes, nome_arquivo: str, modo: str = "completa",
                             calibracao: float = 440.0, nota_ref: str | None = None):
    """Roda a análise completa (ou de cover) e devolve um dicionário pronto pra virar JSON.
    Inclui a curva de pitch já decimada (no máx. 300 pontos) para o gráfico do front-end."""
    audio, sr = carregar_audio_bytes(dados, nome_arquivo)
    if audio is None:
        return {"erro": "Não foi possível ler o áudio. Tente outro formato (WAV ou MP3)."}

    aviso_duracao = None
    duracao_total = len(audio) / sr
    if duracao_total > LIMITE_ANALISE_S:
        audio = audio[: int(LIMITE_ANALISE_S * sr)]
        aviso_duracao = (f"Gravação de {duracao_total:.0f}s: analisei os primeiros "
                         f"{LIMITE_ANALISE_S}s (limite do servidor gratuito).")

    tempos, f0 = extrair_pitch(audio, sr)
    f0_limpo = np.where((f0 >= 80) & (f0 <= 1000), f0, 0.0)

    # Curva de pitch decimada para o gráfico (evita mandar milhares de pontos pro navegador)
    mascara_voz = f0_limpo > 0
    tempos_voz = tempos[mascara_voz]
    f0_voz = f0_limpo[mascara_voz]
    passo = max(1, len(tempos_voz) // 300)
    curva_pitch = [
        {"t": round(float(t), 3), "hz": round(float(f), 2)}
        for t, f in zip(tempos_voz[::passo], f0_voz[::passo])
    ]

    if modo == "cover":
        cover = analisar_cover(audio, sr, calibracao)
        liberar_memoria()
        return {"modo": "cover", "resultado": cover, "curva_pitch": curva_pitch,
                "aviso_duracao": aviso_duracao}

    resultado = analisar_afinacao(f0_limpo, tempos, calibracao, nota_ref=nota_ref)
    seq_notas = extrair_sequencia_notas(f0_limpo, tempos)
    vibratos = detectar_vibrato_v4(f0_limpo, tempos, calibracao_a4=calibracao)
    vibratos_json = [
        {**v, "classificacao": classificar_vibrato_v4(v["taxa_hz"], v["extensao_cents"], v["deslize_cents"], v["periodicidade"])}
        for v in vibratos
    ]
    liberar_memoria()
    return {
        "modo": "completa",
        "resultado": resultado,
        "sequencia_notas": [{"nota": n, "duracao_s": d} for n, d in seq_notas[:20]],
        "vibratos": vibratos_json,
        "curva_pitch": curva_pitch,
        "aviso_duracao": aviso_duracao,
    }



# ══════════════════ AQUECIMENTO (só no Render) ══════════════════
def aquecer_bibliotecas():
    """A 1ª análise depois de o servidor ligar gasta a maior parte do tempo
    carregando/compilando numba, scipy e cia (medido: 18 s de 19 s). Rodando uma
    análise mínima em segundo plano logo na inicialização, quem chegar depois
    já encontra tudo pronto."""
    try:
        t = np.linspace(0, 1.5, int(22050 * 1.5), endpoint=False)
        tom = (0.3 * np.sin(2 * np.pi * 220 * t)).astype(np.float32)
        for passo in (lambda: extrair_pitch(tom, 22050),
                      lambda: detectar_tom(tom, 22050),
                      lambda: detectar_bpm_e_beats(tom, 22050)):
            try:
                passo()
            except Exception:
                pass
    finally:
        liberar_memoria()



Writing /content/audio_analysis.py


In [4]:
%%writefile /content/stems.py
# ══════════════════════════════════════════════════════════════
# stems.py — Separação de voz/instrumental rodando localmente com Demucs
# (biblioteca `demucs`, modelo htdemucs). Não depende mais de um servidor
# Colab externo/gradio.live — a separação acontece no mesmo processo do
# backend, direto no Orange Studio.
#
# Aviso honesto: sem GPU dedicada, cada separação pode levar de dezenas de
# segundos a poucos minutos, dependendo da duração do áudio e do hardware
# onde o backend está rodando.
# ══════════════════════════════════════════════════════════════
import io
import os
import tempfile
import gc


def _carregar_modelo():
    # Propositalmente SEM cache entre chamadas — em um host com pouca memória
    # (512MB), manter o modelo carregado pra sempre economiza tempo mas nunca
    # libera essa memória de volta, mesmo quando ninguém está usando o
    # Estúdio. Recarregar a cada vez custa alguns segundos extras, mas evita
    # que o processo inteiro seja reiniciado por estourar o limite de RAM.
    from demucs.pretrained import get_model
    modelo = get_model("htdemucs")
    modelo.eval()
    return modelo


def _dispositivo():
    """GPU se houver (Colab com GPU é bem mais rápido); senão CPU."""
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


def separar(dados_audio, nome_arquivo):
    """Retorna (voz_bytes, instrumental_bytes, None) em sucesso,
    ou (None, None, mensagem_de_erro) em falha."""
    if not dados_audio:
        return None, None, "Grave ou suba um áudio primeiro."

    try:
        import torch
        import torchaudio
        from demucs.apply import apply_model
    except Exception as e:
        return None, None, (
            "A separação de stems não está disponível nesta hospedagem gratuita "
            "(o Demucs precisa de mais memória do que os 512 MB do plano). "
            "Use o notebook do Colab para separar os stems, ou hospede num plano com mais RAM."
        )

    caminho_temp = None
    try:
        modelo = _carregar_modelo()

        sufixo = os.path.splitext(nome_arquivo or "")[1] or ".wav"
        fd, caminho_temp = tempfile.mkstemp(suffix=sufixo)
        with os.fdopen(fd, "wb") as f:
            f.write(dados_audio)

        wav, sr = torchaudio.load(caminho_temp)
        if sr != modelo.samplerate:
            wav = torchaudio.functional.resample(wav, sr, modelo.samplerate)
        if wav.shape[0] == 1:
            wav = wav.repeat(2, 1)
        elif wav.shape[0] > 2:
            wav = wav[:2]

        referencia = wav.mean(0)
        media, desvio = referencia.mean(), referencia.std() + 1e-8
        wav_normalizado = (wav - media) / desvio

        with torch.no_grad():
            dispositivo = _dispositivo()
            try:
                fontes = apply_model(modelo, wav_normalizado[None], device=dispositivo, progress=False)[0]
            except Exception:
                if dispositivo == "cpu":
                    raise
                fontes = apply_model(modelo, wav_normalizado[None], device="cpu", progress=False)[0]
        fontes = fontes * desvio + media

        nomes_fontes = modelo.sources  # ex.: ['drums', 'bass', 'other', 'vocals']
        indice_voz = nomes_fontes.index("vocals")
        voz = fontes[indice_voz]
        instrumental = sum(fontes[i] for i in range(len(nomes_fontes)) if i != indice_voz)

        # Salva em ARQUIVOS temporários de verdade (não BytesIO). Versões recentes
        # do torchaudio usam o backend "torchcodec", que não lida bem com destinos
        # em memória mesmo com format= explícito — só é confiável com um caminho
        # de arquivo real com extensão.
        fd_voz, caminho_voz = tempfile.mkstemp(suffix=".wav")
        os.close(fd_voz)
        fd_inst, caminho_inst = tempfile.mkstemp(suffix=".wav")
        os.close(fd_inst)
        try:
            torchaudio.save(caminho_voz, voz.clamp(-1, 1), modelo.samplerate, format="wav")
            torchaudio.save(caminho_inst, instrumental.clamp(-1, 1), modelo.samplerate, format="wav")
            with open(caminho_voz, "rb") as f:
                voz_bytes = f.read()
            with open(caminho_inst, "rb") as f:
                inst_bytes = f.read()
        finally:
            for c in (caminho_voz, caminho_inst):
                if os.path.exists(c):
                    os.remove(c)

        return voz_bytes, inst_bytes, None
    except Exception as e:
        return None, None, f"Não foi possível separar os stems: {e}"
    finally:
        if caminho_temp and os.path.exists(caminho_temp):
            os.remove(caminho_temp)
        # Solta as referências aos tensores/modelo pesados e força a coleta de
        # lixo — em host com pouca RAM, isso é a diferença entre o processo
        # sobreviver ou ser reiniciado por estourar o limite de memória.
        modelo = wav = wav_normalizado = fontes = voz = instrumental = referencia = None
        gc.collect()


Writing /content/stems.py


In [5]:
%%writefile /content/separador_worker.py
# ══════════════════════════════════════════════════════════════
# separador_worker.py — roda DENTRO do Google Colab (notebook separador_stems).
# Fica em laço: avisa que está online (batida de coração), procura trabalhos
# "pendente" no Firestore, baixa o áudio do Storage, separa com o Demucs,
# converte os stems pra MP3 (leve) e devolve. O app (Render) só lê o resultado.
# Toda a lógica recebe db/bucket/funções por parâmetro pra poder ser testada
# sem Firebase de verdade.
# ══════════════════════════════════════════════════════════════
import io
import time
from datetime import datetime, timezone

COLECAO_JOBS = "estudio_jobs"
COLECAO_STATUS = "estudio_status"
DOC_STATUS = "colab"


def _agora():
    return datetime.now(timezone.utc).isoformat()


def wav_para_mp3(wav_bytes, kbps=192):
    """WAV (16/32 bits, mono/estéreo) -> MP3. Os stems em WAV pesam ~30 MB por
    música; em MP3 ficam em ~4 MB, o que importa pra subir/baixar rápido."""
    import numpy as np
    import soundfile as sf
    import lameenc
    dados, sr = sf.read(io.BytesIO(wav_bytes), dtype="int16", always_2d=True)
    enc = lameenc.Encoder()
    enc.set_bit_rate(kbps)
    enc.set_in_sample_rate(int(sr))
    enc.set_channels(int(dados.shape[1]))
    enc.set_quality(2)
    return bytes(enc.encode(np.ascontiguousarray(dados).tobytes()) + enc.flush())


def bater_coracao(db, online=True, gpu=False):
    db.collection(COLECAO_STATUS).document(DOC_STATUS).set(
        {"online": bool(online), "atualizado_em": _agora(), "gpu": bool(gpu)}
    )


def processar_um_job(db, bucket, separar_fn, analisar_fn=None, log=print):
    """Pega UM trabalho pendente e processa. Devolve True se havia trabalho."""
    achados = list(db.collection(COLECAO_JOBS).where("status", "==", "pendente").limit(1).stream())
    if not achados:
        return False

    job_id = achados[0].id
    dados_job = achados[0].to_dict() or {}
    ref = db.collection(COLECAO_JOBS).document(job_id)
    nome = dados_job.get("nome_original") or "audio.wav"
    log(f"▶ Trabalho {job_id[:8]}… ({nome}) — separando…")
    inicio = time.time()

    ref.update({"status": "processando", "atualizado_em": _agora()})
    try:
        entrada = bucket.blob(dados_job["entrada_path"]).download_as_bytes()
        voz_wav, inst_wav, erro = separar_fn(entrada, nome)
        if erro:
            raise RuntimeError(erro)

        voz_mp3 = wav_para_mp3(voz_wav)
        inst_mp3 = wav_para_mp3(inst_wav)
        caminho_voz = f"estudio/{job_id}/voz.mp3"
        caminho_inst = f"estudio/{job_id}/instrumental.mp3"
        bucket.blob(caminho_voz).upload_from_string(voz_mp3, content_type="audio/mpeg")
        bucket.blob(caminho_inst).upload_from_string(inst_mp3, content_type="audio/mpeg")

        bpm = tom = None
        if analisar_fn is not None:
            try:
                bpm, tom = analisar_fn(entrada, nome)
            except Exception:
                pass  # BPM/tom são só um extra — nunca derrubam a separação

        ref.update({
            "status": "pronto", "atualizado_em": _agora(),
            "voz_path": caminho_voz, "inst_path": caminho_inst,
            "bpm": bpm, "tom": tom,
        })
        log(f"✅ Pronto em {time.time() - inicio:.0f} s")
    except Exception as e:
        ref.update({"status": "erro", "erro": str(e)[:300], "atualizado_em": _agora()})
        log(f"❌ Falhou: {e}")
    return True


def rodar(db, bucket, separar_fn, analisar_fn=None, gpu=False,
          intervalo_s=4, batida_s=20, log=print, parar=lambda: False):
    """Laço principal. `parar` existe só pra testes."""
    log("🟢 Separador ligado. Deixe esta célula rodando e volte ao Orange Harmony.")
    ultima_batida = 0.0
    try:
        while not parar():
            if time.time() - ultima_batida >= batida_s:
                bater_coracao(db, True, gpu)
                ultima_batida = time.time()
            try:
                if processar_um_job(db, bucket, separar_fn, analisar_fn, log):
                    ultima_batida = 0.0      # avisa "online" logo depois de um trabalho longo
                    continue
            except Exception as e:
                log(f"⚠️ Erro ao consultar a fila (tento de novo): {e}")
            time.sleep(intervalo_s)
    finally:
        try:
            bater_coracao(db, False, gpu)
            log("⚪ Separador desligado.")
        except Exception:
            pass


Writing /content/separador_worker.py


## 3. Credenciais do Firebase

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_APPLICATION_CREDENTIALS_JSON"] = userdata.get("FIREBASE_JSON")
    print("Credenciais lidas dos Segredos do Colab.")
except Exception:
    from google.colab import files
    print("Selecione o arquivo firebase_service_account.json:")
    enviados = files.upload()
    os.environ["GOOGLE_APPLICATION_CREDENTIALS_JSON"] = open(list(enviados.keys())[0], encoding="utf-8").read()
    print("Credenciais carregadas.")

os.environ["FIREBASE_STORAGE_BUCKET"] = "orange-harmony---vocal-ai.firebasestorage.app"

Selecione o arquivo firebase_service_account.json:


## 4. Ligar o separador (deixe esta célula rodando)

In [ ]:
import sys
sys.path.insert(0, "/content")

import firebase_config
db, bucket = firebase_config.get_db(), firebase_config.get_bucket()
if db is None or bucket is None:
    raise RuntimeError(firebase_config.get_erro() or "Firebase não configurado.")

import torch, stems, audio_analysis, separador_worker

gpu = torch.cuda.is_available()
print("Processador:", "GPU (rápido)" if gpu else "CPU (funciona, mas é mais lento — veja a dica da GPU acima)")


def analisar(dados, nome):
    """BPM e tom da música original (só um extra pra mostrar no app)."""
    audio, sr = audio_analysis.carregar_audio_bytes(dados, nome)
    bpm, _ = audio_analysis.detectar_bpm_e_beats(audio, sr)
    return bpm, audio_analysis.detectar_tom(audio, sr)


separador_worker.rodar(db, bucket, stems.separar, analisar, gpu=gpu)